### Objective

The objective of this step is to use BERT-based sentence embeddings to group customer complaints based on semantic similarity. This step serves as a deep learning based alternative to classical topic modeling and allows comparison between statistical and embedding based approaches.


In [1]:
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [8]:
df = pd.read_csv("../data/processed/complaints_step2_cleaned.csv")

df_sample = df.sample(n=20000, random_state=42).reset_index(drop=True)
df_sample.shape

(20000, 7)

In [9]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
embeddings = model.encode(
    df_sample["clean_text"].tolist(),
    batch_size=32,
    show_progress_bar=True
)

Batches:   0%|          | 0/625 [00:00<?, ?it/s]

In [11]:
n_clusters = 8
kmeans = KMeans(
    n_clusters=n_clusters,
    random_state=42,
    n_init=10
)

cluster_labels = kmeans.fit_predict(embeddings)

In [12]:
df_sample["bert_cluster"] = cluster_labels
df_sample.head()

,Date received,Consumer complaint narrative,Product,Issue,Sub-issue,Company,clean_text,bert_cluster
0,2018-11-07,I purchased a car of the year from XXXX in XX/...,Vehicle loan or lease,Problems at the end of the loan or lease,Unable to receive car title or other problem a...,SANTANDER CONSUMER USA HOLDINGS INC.,i purchased a car of the year from in / /20...,4
1,2018-02-27,In XXXX my mortgage servicer was XXXX XXXX XXX...,Mortgage,Struggling to pay mortgage,NaN,Specialized Loan Servicing LLC,in my mortgage servicer was . i was goi...,1
2,2019-01-18,"In XX/XX/XXXX, I applied for one of the two lo...",Student loan,Dealing with your lender or servicer,Trouble with how payments are being handled,SLM CORPORATION,"in / / , i applied for one of the two loans f...",4
3,2016-04-18,I have been trying to work with my mortgage le...,Mortgage,"Loan modification,collection,foreclosure",NaN,PROVIDENT FUNDING ASSOCIATES,i have been trying to work with my mortgage le...,1
4,2015-05-20,Specialized Loan Services acquired my loan fro...,Mortgage,"Loan servicing, payments, escrow account",NaN,Specialized Loan Servicing LLC,specialized loan services acquired my loan fro...,1


In [13]:
def show_cluster_examples(df, cluster_id, n_examples=5):
    examples = df[df["bert_cluster"] == cluster_id]["clean_text"].head(n_examples)
    print(f"\nCluster {cluster_id} examples:")
    for text in examples:
        print("-", text[:200])


In [14]:
for i in range(n_clusters):
    show_cluster_examples(df_sample, i)


Cluster 0 examples:
- the original debt was to hsbc i settled the debt on  / /  with                   and have a letter from them that debt is paid in full acct #   and have spoken with them and they say they own the debt
- i took out an unsecured loan with citibank back in 2008 and a year later, i was working part time and did n't have enough to pay each month. i called up citibank and negotiated with them and they refe
- collection account remain on credit bureau report after 50 day processing dispute reviewed by fdcpa along with collection   submitted in violation fdcpa violations remove from all   credit reporting a
- " collection agency '' - crown asset management llc.               georgia   telephone #    . has filed for the second time, an wrongful lawsuit against myself (     ) of         florida   - telephone
- this is the third time i have had a collection agency contact me regarding a debt i paid in  / /2012. the original debt was with citibank. i settled with them for an

In [15]:
silhouette = silhouette_score(embeddings, cluster_labels)
silhouette

0.037426002

## Summary

In this step, a deep learning based approach was explored using BERT sentence embeddings to group customer complaints based on their overall meaning. Complaint text was converted into semantic vectors and clustered using K-Means.

This step was mainly used to compare an embedding based approach with the LDA topic modeling used earlier. While the clusters captured semantic similarity, they were harder to interpret and not suitable for time based trend analysis. For this reason, BERT clustering was kept as a supporting analysis rather than the main method.
